### bổ sung các thứ cần tiết cho data văn hóa

In [1]:
import json, hashlib, datetime, uuid

# --- Đường dẫn file ---
input_path = "data/culture/culture_vi_full.json"                 # file gốc
output_path = "data/culture/culture_vi_enriched.json"       # file sau khi bổ sung

# --- Hàm tạo checksum để phát hiện trùng lặp ---
def checksum(text: str):
    return hashlib.sha1(text.encode("utf-8")).hexdigest()

# --- Hàm bổ sung thông tin cho từng bản ghi ---
def enrich_record(rec):
    now = datetime.datetime.now().strftime("%Y-%m-%d")

    # ID duy nhất
    rec["id"] = rec.get("id") or str(uuid.uuid4())

    # Ngôn ngữ
    rec["language"] = rec.get("language", "vi")

    # Bản quyền và nguồn
    rec["license"] = rec.get("license", "CC-BY-SA-4.0")
    rec["license_url"] = rec.get("license_url", "https://creativecommons.org/licenses/by-sa/4.0/")
    rec["source_type"] = rec.get("source_type", "wikipedia")

    # Thông tin ngày tháng
    rec["publish_date"] = rec.get("publish_date", now)
    rec["last_updated"] = rec.get("last_updated", now)

    # Provenance: ghi lại công cụ, ngày crawl
    rec["provenance"] = rec.get(
        "provenance",
        {"crawler": "auto_enrich_v1", "crawl_date": now}
    )

    # Giữ nguyên summary/description, KHÔNG sinh tự động
    if rec.get("summary"):
        pass
    elif rec.get("description") and len(rec["description"]) > 100:
        rec["summary"] = rec["description"]
    else:
        rec["summary"] = None

    # Tag và entity để trống nếu chưa có
    rec["tags"] = rec.get("tags", [])
    rec["entities"] = rec.get("entities", [])

    # Địa điểm và toạ độ
    rec["location"] = rec.get("location", "Việt Nam")
    rec["geo"] = rec.get("geo", None)

    # Làm sạch nội dung (bỏ \n và khoảng trắng thừa)
    content_text = rec.get("content", "")
    rec["clean_content"] = rec.get("clean_content") or content_text.replace("\n", " ").strip()

    # Hash SHA1 cho kiểm tra trùng lặp
    rec["checksum"] = checksum(rec["clean_content"])

    # Placeholder cho pipeline RAG
    rec["chunk_id"] = rec.get("chunk_id", None)
    rec["embedding_meta"] = rec.get("embedding_meta", {"model": None, "vector_id": None})

    return rec


# --- Đọc file gốc ---
with open(input_path, "r", encoding="utf-8") as f:
    data = json.load(f)

# --- Áp dụng enrich cho từng bản ghi ---
enriched = [enrich_record(rec) for rec in data]

# --- Ghi ra file mới ---
with open(output_path, "w", encoding="utf-8") as f:
    json.dump(enriched, f, ensure_ascii=False, indent=2)

print(f"Đã bổ sung metadata. File lưu tại: {output_path}")


Đã bổ sung metadata. File lưu tại: data/culture/culture_vi_enriched.json
